In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from itertools import product
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from src.models.rnn import RNN
from src.models.gru import GRU
from src.models.lstm import LSTM
from src.data_loader import load_symbol
from src.features import build_features
from src.cpo import CPO
from src.backtest import Backtest

L = 30
HIDDEN = 32
SEEDS = [0, 1, 2]
MODELS = {'RNN': RNN, 'GRU': GRU, 'LSTM': LSTM}

THRESHOLDS = [0.5, 1.0, 1.5, 2.0]
WINDOWS = [5, 10, 20, 30]
COMBINATIONS = list(product(THRESHOLDS, WINDOWS))

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
es_df = load_symbol('ES=F', start='2005-01-01', end='2026-01-01')
es_features = build_features(es_df)

es_features['y'] = es_features['log_return'].shift(-1)
es_features = es_features.replace([np.inf, -np.inf], np.nan).dropna()

y = es_features['y']
X = es_features.drop(columns=['y'])

n = len(X)
VAL_LEN = 250
TEST_LEN = 250

FOLDS = []
t = int(n * 0.4)
while t + VAL_LEN + TEST_LEN <= n:
    FOLDS.append((t, t + VAL_LEN, t + VAL_LEN + TEST_LEN))
    t += TEST_LEN

print(n, len(FOLDS))

5266 11


/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/Users/sergheibi

In [4]:
def make_windows(sx, sy, X_part, y_part, L):
    Xs = sx.transform(X_part)
    ys = sy.transform(y_part.values.reshape(-1, 1)).ravel()
    xw, yw = [], []
    for i in range(L, len(Xs) + 1):
        xw.append(Xs[i - L:i])
        yw.append(ys[i - 1])
    xw = torch.tensor(np.array(xw), dtype=torch.float32)
    yw = torch.tensor(np.array(yw), dtype=torch.float32).unsqueeze(1)
    return xw, yw, X_part.index[L - 1:]

def train_model(model_cls, seed, Xtr, ytr, Xva, yva, epochs=60, patience=6, lr=1e-3, wd=1e-4, bs=64):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = model_cls(input_size=Xtr.shape[2], hidden_size=HIDDEN, output_size=1)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loss_fn = nn.MSELoss()
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=bs, shuffle=True)
    best_val, best_state, bad = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xva), yva).item()
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model

def predict(model, sy, Xw, dates):
    with torch.no_grad():
        p = model(Xw).numpy()
    return pd.Series(sy.inverse_transform(p).ravel(), index=dates, name='prediction')

def select_and_test(pred_val, pred_test, actual_val, actual_test):
    val_table = CPO(COMBINATIONS, pred_val, actual_val).evaluate()
    best = val_table.iloc[0]
    best_threshold = float(best['zscore_threshold'])
    best_window = int(best['zscore_window'])
    back = Backtest(pred_test, actual_test, best_threshold, best_window)
    return back.order()['strategy_return'].iloc[best_window:]

def save_returns(strategy_returns, name):
    out = strategy_returns.rename('return').rename_axis('date').reset_index()
    out.to_csv(OUTPUT_DIR / f'strategy_returns_{name}.csv', index=False)

In [5]:
collected = {}

for seed in SEEDS:
    for fold, (a, b, c) in enumerate(FOLDS):
        sx = StandardScaler().fit(X.iloc[:a])
        sy = StandardScaler().fit(y.iloc[:a].values.reshape(-1, 1))

        Xtr, ytr, _ = make_windows(sx, sy, X.iloc[:a], y.iloc[:a], L)
        Xva, yva, dates_val = make_windows(sx, sy, X.iloc[a - L + 1:b], y.iloc[a - L + 1:b], L)
        Xte, yte, dates_test = make_windows(sx, sy, X.iloc[b - L + 1:c], y.iloc[b - L + 1:c], L)

        actual_val = y.loc[dates_val]
        actual_test = y.loc[dates_test]

        for name, cls in MODELS.items():
            model = train_model(cls, seed, Xtr, ytr, Xva, yva)
            pv = predict(model, sy, Xva, dates_val)
            pt = predict(model, sy, Xte, dates_test)
            sr = select_and_test(pv, pt, actual_val, actual_test)
            collected.setdefault((name, seed), []).append(sr)

        rng = np.random.default_rng(seed * 1000 + fold)
        pv = pd.Series(rng.standard_normal(len(dates_val)), index=dates_val)
        pt = pd.Series(rng.standard_normal(len(dates_test)), index=dates_test)
        sr = select_and_test(pv, pt, actual_val, actual_test)
        collected.setdefault(('random', seed), []).append(sr)

        print(seed, fold, 'done')

oos = {k: pd.concat(v).sort_index() for k, v in collected.items()}
for (name, seed), r in oos.items():
    if seed == SEEDS[0]:
        save_returns(r, name)

0 0 done
0 1 done
0 2 done
0 3 done
0 4 done
0 5 done
0 6 done
0 7 done
0 8 done
0 9 done
0 10 done
1 0 done
1 1 done
1 2 done
1 3 done
1 4 done
1 5 done
1 6 done
1 7 done
1 8 done
1 9 done
1 10 done
2 0 done
2 1 done
2 2 done
2 3 done
2 4 done
2 5 done
2 6 done
2 7 done
2 8 done
2 9 done
2 10 done


In [6]:
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(252) if r.std() > 0 else 0.0

runs = pd.DataFrame([
    {'model': k[0], 'seed': k[1], 'sharpe': sharpe(r), 'total_return': (1 + r).prod() - 1, 'days': len(r)}
    for k, r in oos.items()
])

summary = runs.groupby('model').agg(
    sharpe_mean=('sharpe', 'mean'),
    sharpe_std=('sharpe', 'std'),
    total_return_mean=('total_return', 'mean'),
    days=('days', 'first'),
)

rand_median = runs[runs['model'] == 'random']['sharpe'].median()
summary['beat_random_median'] = runs.groupby('model')['sharpe'].apply(lambda s: (s > rand_median).mean())
summary.round(4)

,sharpe_mean,sharpe_std,total_return_mean,days,beat_random_median
model,,,,,
GRU,0.6327,0.0207,0.9793,2600,0.6667
LSTM,0.2314,0.1422,0.2088,2595,0.0000
RNN,0.3067,0.2875,0.3442,2550,0.0000
random,0.5326,0.2203,0.5504,2585,0.3333


In [7]:
OUTPUT_DIR = Path('outputs').resolve()
OUTPUT_DIR.mkdir(exist_ok=True)

for (name, seed), r in oos.items():
    if seed == SEEDS[0]:
        out = r.rename('return').rename_axis('date').reset_index()
        path = OUTPUT_DIR / f'strategy_returns_{name}.csv'
        out.to_csv(path, index=False)
        print(path, len(out))

/Users/sergheibigboy/cpo_rnn/outputs/strategy_returns_RNN.csv 2550
/Users/sergheibigboy/cpo_rnn/outputs/strategy_returns_GRU.csv 2600
/Users/sergheibigboy/cpo_rnn/outputs/strategy_returns_LSTM.csv 2595
/Users/sergheibigboy/cpo_rnn/outputs/strategy_returns_random.csv 2585
